In [1]:
# ===== CELL 1: Setup + Mount Drive =====
from google.colab import drive
drive.mount('/content/drive')

import os

base_path = "/content/drive/MyDrive/hadoop_openstack/OpenStack"

files = {
    "labels": os.path.join(base_path, "anomaly_labels.txt"),
    "abnormal": os.path.join(base_path, "openstack_abnormal.log"),
    "normal1": os.path.join(base_path, "openstack_normal1.log"),
    "normal2": os.path.join(base_path, "openstack_normal2.log"),
}

# File sizes check (RAM planning ke liye)
for name, path in files.items():
    size_mb = os.path.getsize(path) / (1024*1024)
    print(f"{name}: {size_mb:.2f} MB")

Mounted at /content/drive
labels: 0.00 MB
abnormal: 5.20 MB
normal1: 14.83 MB
normal2: 38.76 MB


In [4]:
import os

# Pehle check karo clone hua?
print("Root files:")
print(os.listdir("/content/ASAD-thesis-files"))

print("\nData folder:")
print(os.listdir("/content/ASAD-thesis-files/data"))

print("\nRaw folder:")
print(os.listdir("/content/ASAD-thesis-files/data/raw"))

Root files:


FileNotFoundError: [Errno 2] No such file or directory: '/content/ASAD-thesis-files'

In [5]:
!git clone https://github.com/asadkhanofficial2341-beep/ASAD-thesis-files.git
%cd ASAD-thesis-files
!ls

Cloning into 'ASAD-thesis-files'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 50 (delta 12), reused 46 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (50/50), 11.25 MiB | 13.89 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/ASAD-thesis-files
data  docs  notebooks  results


In [6]:
import zipfile, os

zip_path = "data/raw/hadoop_openstack/Hadoop.zip"
extract_path = "/content/hadoop_data"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction done!")
print("Files:", os.listdir(extract_path)[:10])

Extraction done!
Files: ['application_1445144423722_0021', 'application_1445076437777_0004', 'application_1445087491445_0007', 'application_1445182159119_0001', 'application_1445144423722_0023', 'application_1445062781478_0016', 'application_1445182159119_0002', 'application_1445182159119_0005', 'application_1445062781478_0019', 'application_1445094324383_0002']


In [7]:
os.makedirs("notebooks/hadoop", exist_ok=True)
os.makedirs("results/hadoop", exist_ok=True)
print("Folders created!")

Folders created!


In [8]:
with open(f"{extract_path}/abnormal_label.txt", "r") as f:
    print(f.read())

The logs are generated from a Hadoop cluster using two testing applications: WordCount and PageRank. Each application has been run for several times, simulating both normal and abnormal cases with injected specific failures.

### WordCount
Normal:
+ application_1445087491445_0005
+ application_1445087491445_0007
+ application_1445175094696_0005

Machine down:
+ application_1445087491445_0001
+ application_1445087491445_0002
+ application_1445087491445_0003
+ application_1445087491445_0004
+ application_1445087491445_0006
+ application_1445087491445_0008
+ application_1445087491445_0009
+ application_1445087491445_0010
+ application_1445094324383_0001
+ application_1445094324383_0002
+ application_1445094324383_0003
+ application_1445094324383_0004
+ application_1445094324383_0005

Network disconnection:
+ application_1445175094696_0001
+ application_1445175094696_0002
+ application_1445175094696_0003
+ application_1445175094696_0004

Disk full:
+ application_1445182159119_0001
+ applic

In [9]:
import re

with open(f"{extract_path}/abnormal_label.txt", "r") as f:
    label_text = f.read()

label_dict = {}
current_category = None

for line in label_text.split("\n"):
    line = line.strip()
    if line.startswith("Normal"):
        current_category = "Normal"
    elif line.startswith("Machine down"):
        current_category = "Machine down"
    elif line.startswith("Network disconnection"):
        current_category = "Network disconnection"
    elif line.startswith("Disk full"):
        current_category = "Disk full"
    elif line.startswith("+"):
        app_id = line.replace("+", "").strip()
        label_dict[app_id] = current_category

print(f"Total labeled applications: {len(label_dict)}")
print(dict(list(label_dict.items())[:5]))

Total labeled applications: 55
{'application_1445087491445_0005': 'Normal', 'application_1445087491445_0007': 'Normal', 'application_1445175094696_0005': 'Normal', 'application_1445087491445_0001': 'Machine down', 'application_1445087491445_0002': 'Machine down'}


In [10]:
app_folders = [d for d in os.listdir(extract_path) if os.path.isdir(os.path.join(extract_path, d))]
print(f"Total application folders: {len(app_folders)}")

matched = sum(1 for app in app_folders if app in label_dict)
print(f"Matched with labels: {matched}")

unmatched = [app for app in app_folders if app not in label_dict]
print(f"Unmatched: {unmatched}")

Total application folders: 55
Matched with labels: 55
Unmatched: []


In [11]:
log_files = []

for root, dirs, files in os.walk(extract_path):
    for f in files:
        if f.endswith(".log"):
            log_files.append(os.path.join(root, f))

print(f"Total log files found: {len(log_files)}")
print("Sample:", log_files[0])

Total log files found: 978
Sample: /content/hadoop_data/application_1445144423722_0021/container_1445144423722_0021_01_000013.log


In [12]:
import pandas as pd

records = []

for filepath in log_files:
    parts = filepath.split("/")
    app_id = parts[-2]
    container_id = parts[-1].replace(".log", "")

    category = label_dict.get(app_id, "Unknown")
    binary_label = 0 if category == "Normal" else 1

    try:
        with open(filepath, "r", errors="ignore") as f:
            lines = f.readlines()
    except Exception as e:
        continue

    for line in lines:
        line = line.strip()
        if line:
            records.append({
                "app_id": app_id,
                "container_id": container_id,
                "raw_line": line,
                "failure_type": category,
                "label": binary_label
            })

df = pd.DataFrame(records)
print(f"Total lines loaded: {len(df)}")
df.head()

Total lines loaded: 393433


,app_id,container_id,raw_line,failure_type,label
0,application_1445144423722_0021,container_1445144423722_0021_01_000013,"2015-10-18 18:04:15,524 INFO [main] org.apache...",Normal,0
1,application_1445144423722_0021,container_1445144423722_0021_01_000013,"2015-10-18 18:04:15,649 INFO [main] org.apache...",Normal,0
2,application_1445144423722_0021,container_1445144423722_0021_01_000013,"2015-10-18 18:04:15,649 INFO [main] org.apache...",Normal,0
3,application_1445144423722_0021,container_1445144423722_0021_01_000013,"2015-10-18 18:04:15,664 INFO [main] org.apache...",Normal,0
4,application_1445144423722_0021,container_1445144423722_0021_01_000013,"2015-10-18 18:04:15,664 INFO [main] org.apache...",Normal,0


In [ ]:
print("Binary label distribution:")
print(df['label'].value_counts())
print(f"\nAnomaly %: {df['label'].mean()*100:.2f}%")

print("\nFailure type distribution:")
print(df['failure_type'].value_counts())

In [13]:
from drain3 import TemplateMiner
import os

os.makedirs("/content/drive/MyDrive/ASAD_Thesis/hadoop_results", exist_ok=True)

print("Drain3 parsing shuru...")
miner = TemplateMiner()

templates = []
cluster_ids = []

for line in df['raw_line'].astype(str):
    result = miner.add_log_message(line)
    templates.append(result['template_mined'])
    cluster_ids.append(result['cluster_id'])

df['template'] = templates
df['cluster_id'] = cluster_ids

print(f"Drain3 complete!")
print(f"Unique templates: {len(miner.drain.clusters)}")
print(df[['raw_line', 'template', 'cluster_id', 'label']].head(3))

ModuleNotFoundError: No module named 'drain3'

In [14]:
!pip install drain3 -q

  Preparing metadata (setup.py) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyiceberg 0.11.1 requires cachetools<7.0,>=5.5, but you have cachetools 4.2.1 which is incompatible.


In [15]:
from drain3 import TemplateMiner
import os

os.makedirs("/content/drive/MyDrive/ASAD_Thesis/hadoop_results", exist_ok=True)

print("Drain3 parsing shuru...")
miner = TemplateMiner()

templates = []
cluster_ids = []

for line in df['raw_line'].astype(str):
    result = miner.add_log_message(line)
    templates.append(result['template_mined'])
    cluster_ids.append(result['cluster_id'])

df['template'] = templates
df['cluster_id'] = cluster_ids

print(f"Drain3 complete!")
print(f"Unique templates: {len(miner.drain.clusters)}")
print(df[['raw_line', 'template', 'cluster_id', 'label']].head(3))

Drain3 parsing shuru...
Drain3 complete!
Unique templates: 702
                                            raw_line  \
0  2015-10-18 18:04:15,524 INFO [main] org.apache...   
1  2015-10-18 18:04:15,649 INFO [main] org.apache...   
2  2015-10-18 18:04:15,649 INFO [main] org.apache...   

                                            template  cluster_id  label  
0  2015-10-18 18:04:15,524 INFO [main] org.apache...           1      0  
1  2015-10-18 18:04:15,649 INFO [main] org.apache...           2      0  
2  2015-10-18 18:04:15,649 INFO [main] org.apache...           3      0  


In [ ]:
import os

# Path change kiya ✅
save_path = "/content/drive/MyDrive/hadoop_openstack/hadoop_logs_parsed.csv"

df.to_csv(save_path, index=False)

size = os.path.getsize(save_path) / (1024*1024)
print(f"Saved! Size: {size:.1f} MB")
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nLabel distribution:")
print(df['label'].value_counts())

In [ ]:
import os

# Drive structure dekho
print(os.listdir("/content/drive/MyDrive"))